# ML-02 — Research Question and Provisional Lane

**Provisional lane:** CTR / Engagement Opportunity Scoring, starting with CTR only.

This notebook frames a review decision and checks whether the starter data supports exploring it. It does not train a model. Prepared with AI assistance; the supporting numbers are calculated from the local starter CSV below.

**Run:** use a Python environment with `pandas` and run all cells. The data loader works from the repository root, starter folder, or notebook folder. References: `docs/ml-intern-dataset-and-lane-guide.md`, `docs/data-dictionary.md`, and `DATA_USE.md` in the starter folder.

## 1. My lane (or freestyle) and why

I choose the predefined **CTR / Engagement Opportunity Scoring** lane. My starting question is about pages that already appear in search but capture few clicks compared with similar pages. It is a simple, explainable direction: the starter dataset contains impressions, clicks, average position, and content type, so I can begin with grouped comparisons and a transparent rule. The counts below show a substantial pool of visible pages to investigate. I will focus on CTR first, keep engagement as possible later context, and revisit this provisional choice by the end of Week 4 if the evidence is weak.

## 2. The question: decision, action, cost of a wrong call

**Research question:** Among pages with at least 500 impressions in the trailing 90 days and a recorded average position from 1 to 20, which pages have CTR below comparable position/content-type peers and should an editor review first?

- **Unit of analysis:** one pseudonymized content item (page) in the starter snapshot, with trailing-90-day measurements. `content_id` is the page key; `client_id` is for grouping and validation.
- **Decision and actor:** a content editor chooses which pages to inspect within a limited review budget. I will initially assume a capacity of 20 pages per review cycle; this is a policy choice to confirm, not a fact supplied by the dataset.
- **Output:** a ranked review queue with a transparent score, observed CTR, peer benchmark, impression count, position band, and reason codes such as `enough_impressions` and `below_peer_ctr`. Pages with missing position or too little evidence go to monitoring rather than being called opportunities.
- **Action:** the editor checks search intent and snippet/metadata quality in an authorized workflow, then decides whether a title/meta or intent-match edit is justified, or whether to monitor. Low CTR alone does not justify rewriting or pruning a page.
- **Cost of a wrong call:** a false positive wastes review/editing time and an unnecessary edit could hurt a useful page. A false negative leaves a possible click opportunity unreviewed. Human review comes before any content change; I cannot price these costs from this CSV.

**Why data helps:** comparing raw CTR across all pages confounds visibility, position, and content type. Data can identify comparable peers and direct scarce review time toward visible pages with a measured gap. My first baseline will estimate peer CTR within position bands and content types, then rank positive gaps weighted by impressions. The resulting score is a review heuristic, not a forecast of recoverable clicks. Sparse peer groups need a minimum-support rule or monitoring, and client mix needs checking.

**Success and validation plan:** compare the peer-gap queue with a simple impressions-only queue at the same capacity. The practical metric is **precision@20**, defined here as the fraction of 20 shortlisted pages that an editor judges worth action under a written review rubric. Those independent editorial labels are absent from the starter CSV, so precision@20 cannot honestly be reported today. The next step is to obtain that review evidence, rather than label pages with my own rule and claim success. I will also check stability when the impression cutoff changes and validate peer benchmarks on held-out clients. If later testing future CTR, I must use separate earlier feature and later outcome windows from the warehouse; this snapshot does not demonstrate future performance.

## 3. Quick look at the data (2–3 real numbers)

I load the bundled CSV without printing individual rows. The checks establish page grain and required fields before computing aggregates. The 500-impression cutoff and positions 1–20 are provisional review policies, not universal SEO thresholds. Rates are stored as percentages: `ctr = 0.5` means **0.5%**.

In [1]:
from pathlib import Path
import pandas as pd

# Find the bundled starter folder from common notebook launch locations.
relative_csv = Path("data/raw/content_refresh_anonymized.csv")
roots = (Path.cwd(), *Path.cwd().parents)
root = next(
    (candidate for base in roots
     for candidate in (base, base / "flyrank-ml-internship-starter-main")
     if (candidate / relative_csv).is_file()),
    None,
)
if root is None:
    raise FileNotFoundError("Run inside a clone containing the starter CSV.")

df = pd.read_csv(root / relative_csv)
required = {"content_id", "client_id", "impressions_90d", "clicks_90d", "avg_position", "ctr", "content_type"}
assert required.issubset(df.columns), "Required columns are missing."
assert df["content_id"].notna().all() and df["content_id"].is_unique, "Expected one row per page."
assert df["client_id"].notna().all(), "Client grouping keys must be present."
assert df[["impressions_90d", "clicks_90d"]].ge(0).all().all(), "Counts must be nonnegative."

eligible = df.loc[
    df["impressions_90d"].ge(500)
    & df["avg_position"].between(1, 20)
].copy()
assert not eligible.empty, "No pages meet the provisional policy."
assert eligible["ctr"].notna().all(), "Eligible pages need measured CTR."
low_ctr = eligible["ctr"].lt(0.5)

print(f"1. Starter coverage: {len(df):,} pages, {df.shape[1]} columns, {df['client_id'].nunique()} clients.")
print(f"2. Review-policy pool: {len(eligible):,} pages ({len(eligible) / len(df):.2%} of the starter), across {eligible['client_id'].nunique()} clients.")
print(f"3. Recorded CTR below 0.5%: {low_ctr.sum():,} eligible pages ({low_ctr.mean():.2%} of that pool).")
print(f"Data-quality context: {df['avg_position'].eq(0).sum():,} pages have position 0, meaning no position data; they are excluded.")

1. Starter coverage: 30,000 pages, 44 columns, 32 clients.
2. Review-policy pool: 12,009 pages (40.03% of the starter), across 28 clients.
3. Recorded CTR below 0.5%: 9,745 eligible pages (81.15% of that pool).
Data-quality context: 1,205 pages have position 0, meaning no position data; they are excluded.


In [2]:
# Descriptive support for position-aware comparisons; no candidate labels.
eligible["position_band"] = pd.cut(
    eligible["avg_position"],
    bins=[0, 3, 10, 20],
    labels=["1–3", ">3–10", ">10–20"],
)
position_summary = eligible.groupby("position_band", observed=True).agg(
    pages=("impressions_90d", "size"),
    impressions=("impressions_90d", "sum"),
    clicks=("clicks_90d", "sum"),
)
# Pooled CTR is computed from total clicks / total impressions, not mean page CTR.
position_summary["pooled_ctr_pct"] = (
    100 * position_summary["clicks"] / position_summary["impressions"]
)
assert position_summary["pages"].sum() == len(eligible)
print(position_summary.round({"pooled_ctr_pct": 3}).to_string())

               pages  impressions  clicks  pooled_ctr_pct
position_band                                            
1–3              466      7436196   36784           0.495
>3–10           7084     88871350  309691           0.348
>10–20          4459     22384996   78544           0.351


**What these numbers suggest:**

1. The starter covers **30,000 pages across 32 clients**. Client differences could matter, so grouping and client holdout belong in later validation.
2. **12,009 pages (40.03% of the starter)** meet the provisional volume/position policy, across 28 clients. This is enough to explore a review queue with a 20-page budget without depending on tiny-volume examples.
3. **9,745 eligible pages (81.15%)** have recorded CTR below 0.5%. A single low-CTR cutoff would flag most of the pool, so it offers little prioritization by itself. This motivates testing peer comparisons; it does not prove that these pages have bad metadata.

The extra position table reports pooled CTR of **0.495%, 0.348%, and 0.351%** for the three bands. The values are not a clean monotonic curve, and each mixes clients and content types. They are descriptive observations, not an established expected-CTR model. I need to inspect those differences before using a benchmark.

The **1,205 zero-position rows** are missing measurements, not pages ranked ahead of position 1. Positive recorded positions below 1 are also excluded by the explicit 1–20 policy and need a data-quality audit. No missing value is silently filled with zero. These findings support exploring this lane over the coming weeks: audit the peer groups, build the rule baseline, validate its review usefulness, and add complexity only if it improves the decision.

## 4. Careful words: what I can and can't claim

**I can claim:** I measured the stated coverage, eligibility counts, and CTR summaries in this starter slice. They suggest that a position-aware review queue is worth investigating. A later score can provide observational decision support, with reasons a human can inspect.

**I cannot claim:** low CTR proves poor titles, that an edit will recover a particular number of clicks, that this starter slice represents all FlyRank clients, that I predict Google's algorithm, or that a refresh causes recovery. The dataset contains no original titles, URLs, search-result context, or independent editorial judgments, so it cannot diagnose the cause of a CTR gap. Average position also hides query/device mix and variation over time.

**Data and leakage boundaries:** publish aggregates only here. Pseudonymous IDs are for keys, joins, grouping, and splits, never model inputs. `trend_direction`, `trend_pct`, and the derived decline label are unnecessary for this CTR framing. A page's current CTR can describe its current gap, but must not serve as both a feature and a claimed future outcome. Any later predictive experiment needs aligned windows and independent observed labels. No dataset, credentials, raw client identifiers, or row-level dump is added to this submission.

## Self-check

- [x] Chose one predefined lane and explained why, provisionally through Week 4.
- [x] Named the research question, page grain, editor decision, output, action, and costs of errors.
- [x] Loaded the starter data and showed at least two real supporting numbers, with code and interpretation.
- [x] Explained why a transparent rule comes first, and what evidence would justify ML later.
- [x] Defined a review metric and stated honestly that its independent labels are not available yet.
- [x] Ran every code cell top to bottom with no errors; outputs are saved in this notebook.
- [x] Checked public outputs: aggregates only, with observed/directional/decision-support language.

**Submission location:** `work/notebooks/w01_research_question.ipynb` inside the starter folder. Commit this executed notebook and submit the repository URL on the ML-02 card.